Copyright Hewlett Packard Enterprise Development LP.


# ORNL Telemetry Data Across All Caps using Arkouda

This is an attempt to transliterate the code in `Data_across_All_Caps.ipynb`
that uses pandas into arkouda without any `for` loops, while trying to achieve
the same results.

This file is sort of a scratchpad. It does not do the full computation that
is in `Data_across_All_Caps.ipynb` and also does some other, extra
calculations that end up not being used.

There is not really any benefit in trying to make sense from this code.

For the final, vectorized arkouda implementation, see `Data_across_all_caps_arkouda_final.ipynb`

In [12]:
import arkouda as ak
ak.connect("x1003c6s1b0n0")

connected to arkouda server tcp://*:5555


In [ ]:
dir = "/lus/scratch/khandeka/hpegithub/ORNL-telemetry-analysis/parquet-traces-for-LSMS-application/"

power_cap200 = "paper_lsms_scorep_papi_wombat_grace_hopper_plugins_production_200_16_events.parquet"
power_cap300 = "paper_lsms_scorep_papi_wombat_grace_hopper_plugins_production_300_16_events.parquet"
power_cap400 = "paper_lsms_scorep_papi_wombat_grace_hopper_plugins_production_400_16_events.parquet"
power_cap500 = "paper_lsms_scorep_papi_wombat_grace_hopper_plugins_production_500_16_events.parquet"
power_cap600 = "paper_lsms_scorep_papi_wombat_grace_hopper_plugins_production_600_16_events.parquet"
power_cap700 = "paper_lsms_scorep_papi_wombat_grace_hopper_plugins_production_700_16_events.parquet"
power_cap800 = "paper_lsms_scorep_papi_wombat_grace_hopper_plugins_production_800_16_events.parquet"
power_cap900 = "paper_lsms_scorep_papi_wombat_grace_hopper_plugins_production_900_16_events.parquet"
power_cap1000 = "paper_lsms_scorep_papi_wombat_grace_hopper_plugins_production_1000_16_events.parquet"

In [14]:
ak_df = ak.read_parquet(dir+power_cap200)
print(ak_df)
full_ak_df = ak.DataFrame(ak_df)


RuntimeError: File /lus/scratch/khandeka/hpegithub/ORNL-telemetry-analysis/parquet-traces-for-LSMS-application-8x/paper_lsms_scorep_papi_wombat_grace_hopper_plugins_production_200_16_events.parquet does not exist in a location accessible to Arkouda

In [ ]:
keys_to_keep = ['time', 'metric', 'values', 'region', 'location', 'event']
ak_df = {key: ak_df[key] for key in keys_to_keep}
print(ak_df)
prim_key = ak.arange(0, len(ak_df['time']), dtype=ak.int64)
ak_df = ak.DataFrame(ak_df)
print(ak_df)

{'time': array(['262148400831264', '262148400831264', '262148400831264', ... , '262741569115744', '262741569154784', '262741586816192']), 'metric': array(['MetricInstance [1]: 'metric_class': MetricClass [0], 'recorder': Location [4294967296] '', 'metric_scope': MetricScope.LOCATION, 'scope': Location [0] 'Master thread'', 'MetricInstance [3]: 'metric_class': MetricClass [2], 'recorder': Location [8589934592] '', 'metric_scope': MetricScope.LOCATION, 'scope': Location [0] 'Master thread'', 'MetricInstance [5]: 'metric_class': MetricClass [4], 'recorder': Location [12884901888] '', 'metric_scope': MetricScope.LOCATION, 'scope': Location [0] 'Master thread'', ... , 'nan', 'nan', 'nan']), 'values': array(['[327000]', '[41904000]', '[44194000]', ... , 'nan', 'nan', 'nan']), 'region': array(['nan', 'nan', 'nan', ... , 'BUFFER FLUSH', 'COMPUTE IDLE', 'nan']), 'location': array(['', '', '', ... , 'Master thread', 'CUDA[0:7]', 'Master thread']), 'event': array(['nan', 'nan', 'nan', ... , 'Leav

In [ ]:
ak_df["time"] = ak_df["time"].astype(ak.int64)
print(ak_df["time"])
ak_df["time"] = (ak_df['time'] - ak_df['time'][0]) / 1000000000
print(ak_df["time"])

[262148400831264 262148400831264 262148400831264 ... 262741569115744 262741569154784 262741586816192]
[0.0 0.0 0.0 ... 593.168 593.168 593.186]


In [ ]:
regions = ak.unique(ak_df['region'])
print(regions)

# Storing the total energy and time for each region in pdarrays
print(f"Number of regions: {len(regions)}")
regions_ndarray = regions.to_ndarray()
for i, region in enumerate(regions_ndarray):
  df_r = ak_df[ak_df['region'] == region]
  print(f"Region: {region}")


['!$omp parallel @calculateDensities.cpp:306', 'cuMemHostAlloc', 'cuStreamCreate', ... , 'cuCtxPopCurrent_v2', 'cuDevicePrimaryCtxRetain', 'trsm_left_kernel<int, double2, 256, 4, true, false, false, false, true>']
Number of regions: 123
Region: !$omp parallel @calculateDensities.cpp:306
Region: cuMemHostAlloc
Region: cuStreamCreate
Region: getrf_pivot<getrf_params_<double2, 512, 3, 512, 256, 1> >
Region: !$omp implicit barrier @energyContourIntegration.cpp:246
Region: copyTMatrixToTauCuda
Region: cuMemFree_v2
Region: cuEventCreate
Region: cuDeviceGetUuid
Region: getrf_pivot<getrf_params_<double2, 512, 2, 512, 64, 8> >
Region: !$omp for @MultipoleMadelung.cpp:105
Region: sm90_xmma_gemm_cf64cf64_f64f64_cf64_nn_n_tilesize32x16x64_stage3_warpsize2x1x2_tensor16x8x16_execute_split_k_kernel__5x_cublas
Region: cuMemsetD8_v2
Region: !$omp implicit barrier @calculateDensities.cpp:335
Region: MPI_Comm_rank
Region: sm90_xmma_gemm_cf64cf64_f64f64_cf64_nn_n_tilesize32x64x32_stage4_warpsize2x4x1_tens

In [ ]:
# 1. Sort by 'region'
sorted_idx = ak.argsort(ak_df['region'])
ak_df_sorted = ak_df[sorted_idx]

# 2. GroupBy to get segments (group boundaries)
gb = ak.GroupBy(ak_df_sorted['region'])
segments, unique_regions = gb.segments, gb.unique_keys

# 3. Iterate over segments (efficient slicing)
for i, region in enumerate(unique_regions.to_ndarray()):
    start = segments[i]
    end = segments[i+1] if i+1 < len(segments) else len(ak_df_sorted)
    df_r = ak_df_sorted[start:end]
    print(f"Region: {region}")


Region: !$omp parallel @calculateDensities.cpp:306
Region: cuMemHostAlloc
Region: cuStreamCreate
Region: getrf_pivot<getrf_params_<double2, 512, 3, 512, 256, 1> >
Region: !$omp implicit barrier @energyContourIntegration.cpp:246
Region: copyTMatrixToTauCuda
Region: cuMemFree_v2
Region: cuEventCreate
Region: cuDeviceGetUuid
Region: getrf_pivot<getrf_params_<double2, 512, 2, 512, 64, 8> >
Region: !$omp for @MultipoleMadelung.cpp:105
Region: sm90_xmma_gemm_cf64cf64_f64f64_cf64_nn_n_tilesize32x16x64_stage3_warpsize2x1x2_tensor16x8x16_execute_split_k_kernel__5x_cublas
Region: cuMemsetD8_v2
Region: !$omp implicit barrier @calculateDensities.cpp:335
Region: MPI_Comm_rank
Region: sm90_xmma_gemm_cf64cf64_f64f64_cf64_nn_n_tilesize32x64x32_stage4_warpsize2x4x1_tensor16x8x16_execute_kernel__5x_cublas
Region: xxtrf4_set_info_ker
Region: cuLibraryGetModule
Region: getrf_pivot<getrf_params_<double2, 128, 1, 128, 128, 1> >
Region: ipiv_lower_diag<double2, 512>
Region: !$omp parallel @calculateTauMatrix

In [ ]:
ak_df_gpupower = ak_df[ak_df["metric"] == "MetricInstance [21]: 'metric_class': MetricClass [20], 'recorder': Location [47244640256] '', 'metric_scope': MetricScope.SYSTEM_TREE_NODE, 'scope': SystemTreeNode [1] 'wombat35'"].copy()
print(ak_df_gpupower["metric"])
print(ak_df_gpupower["values"])
ak_df_gpupower = ak_df_gpupower[ak_df_gpupower["values"] != "nan"]
print(ak_df_gpupower["values"])
print("Separator")
ak_df_gpupower['values'] = ak_df_gpupower['values'].strip('[]').astype(ak.float64) / 1000
print(ak_df_gpupower["values"])
print(ak_df_gpupower)

['MetricInstance [21]: 'metric_class': MetricClass [20], 'recorder': Location [47244640256] '', 'metric_scope': MetricScope.SYSTEM_TREE_NODE, 'scope': SystemTreeNode [1] 'wombat35'', 'MetricInstance [21]: 'metric_class': MetricClass [20], 'recorder': Location [47244640256] '', 'metric_scope': MetricScope.SYSTEM_TREE_NODE, 'scope': SystemTreeNode [1] 'wombat35'', 'MetricInstance [21]: 'metric_class': MetricClass [20], 'recorder': Location [47244640256] '', 'metric_scope': MetricScope.SYSTEM_TREE_NODE, 'scope': SystemTreeNode [1] 'wombat35'', ... , 'MetricInstance [21]: 'metric_class': MetricClass [20], 'recorder': Location [47244640256] '', 'metric_scope': MetricScope.SYSTEM_TREE_NODE, 'scope': SystemTreeNode [1] 'wombat35'', 'MetricInstance [21]: 'metric_class': MetricClass [20], 'recorder': Location [47244640256] '', 'metric_scope': MetricScope.SYSTEM_TREE_NODE, 'scope': SystemTreeNode [1] 'wombat35'', 'MetricInstance [21]: 'metric_class': MetricClass [20], 'recorder': Location [47244

In [ ]:
print(ak_df["region"])
ak_regions = ak_df[ak_df["region"] != "nan"]  # Filter out rows where region is NaN
ak_df_gpuevents = ak_regions[ak_regions["location"] == "CUDA[0:7]"].copy()
ak_df_gpuevents = ak_df_gpuevents[ak_df_gpuevents["region"]!= "nan"].copy()
print(ak_df_gpuevents)

['nan', 'nan', 'nan', ... , 'BUFFER FLUSH', 'COMPUTE IDLE', 'nan']
DataFrame(['time', 'metric', 'values', 'region', 'location', 'event'], 2,405,378 rows, 216327274.00 B)


In [ ]:
df = ak_df_gpuevents
df_power = ak_df_gpupower
print(df)
enter_events = df[df["event"] == "Enter"]
print(enter_events)
print(df[df["event"] == "Enter"])
leave_events = df[df["event"] == "Leave"]
leave_events = leave_events.sort_values("time")
print(leave_events)
print(leave_events["event"])
print(leave_events["time"])

DataFrame(['time', 'metric', 'values', 'region', 'location', 'event'], 2,405,378 rows, 216327274.00 B)
DataFrame(['time', 'metric', 'values', 'region', 'location', 'event'], 1,202,689 rows, 108163637.00 B)
DataFrame(['time', 'metric', 'values', 'region', 'location', 'event'], 1,202,689 rows, 108163637.00 B)
DataFrame(['time', 'metric', 'values', 'region', 'location', 'event'], 1,202,689 rows, 108163637.00 B)
['Leave', 'Leave', 'Leave', ... , 'Leave', 'Leave', 'Leave']
[2.64669 2.64672 2.6507 ... 592.799 592.799 593.168]


In [ ]:
leave_times = leave_events["time"]
print(leave_times)
leave_regions = leave_events["region"]
print(leave_regions)

[2.64669 2.64672 2.6507 ... 592.799 592.799 593.168]
['COMPUTE IDLE', 'setDiagonalKernelCuda<std::complex<double> >', 'COMPUTE IDLE', ... , 'COMPUTE IDLE', 'copyTauToTau00Cuda', 'COMPUTE IDLE']


#### Note on the difference in approach when using Arkouda
Iterating directly over a DataFrame with `for x in df` is not recommended. Doing so is discouraged because it requires transferring all array data from the arkouda server to the Python client since there is almost always a more array-oriented way to express an iterator-based computation.

In [ ]:
import arkouda.array_api as xp

enter_times = enter_events["time"]
print("Enter Times : ", enter_times)
enter_regions = enter_events["region"]
print("Enter Regions : ", enter_regions)
leave_indices = xp.searchsorted(xp.asarray(leave_times), xp.asarray(enter_times), side="left") # Bug in searchsorted. Meaning of side="left" and side="right" is swapped
leave_indices = ak.array(leave_indices)
print("Leave Indices : ", leave_indices)
valid_indices = (leave_indices < len(leave_times)) & (leave_regions[leave_indices] == enter_regions)
valid_indices = (leave_indices < len(leave_times)) & (leave_regions[leave_indices] == enter_regions)


print("Valid Indices : ", valid_indices)

valid_enter_times = enter_times[valid_indices]
print("Valid Enter Times : ", valid_enter_times)
valid_leave_times = leave_times[leave_indices[valid_indices]]
print("Valid Leave Times : ", valid_leave_times)
valid_regions = enter_regions[valid_indices]
print("Valid Regions : ", valid_regions)

durations = valid_leave_times - valid_enter_times
print("Durations : ", durations)

# Create an Arkouda DataFrame
region_times_df = ak.DataFrame({
    'region': valid_regions,
    'duration': durations
})

# Group by 'region' and sum the 'duration'
region_times = region_times_df.groupby('region').sum("duration")

Enter Times :  [0.0520531 2.64669 2.64672 ... 592.799 592.799 592.799]
Enter Regions :  ['COMPUTE IDLE', 'setDiagonalKernelCuda<std::complex<double> >', 'COMPUTE IDLE', ... , 'COMPUTE IDLE', 'copyTauToTau00Cuda', 'COMPUTE IDLE']


/lus/bnchlu1/khandeka/arkouda/arkouda/array_api/__init__.py:287: UserWarning: The arkouda.array_api submodule is still experimental.
  warnings.warn("The arkouda.array_api submodule is still experimental.")


Leave Indices :  [0 1 2 ... 1202686 1202687 1202688]
Valid Indices :  [True True True ... True True True]
Valid Enter Times :  [0.0520531 2.64669 2.64672 ... 592.799 592.799 592.799]
Valid Leave Times :  [2.64669 2.64672 2.6507 ... 592.799 592.799 593.168]
Valid Regions :  ['COMPUTE IDLE', 'setDiagonalKernelCuda<std::complex<double> >', 'COMPUTE IDLE', ... , 'COMPUTE IDLE', 'copyTauToTau00Cuda', 'COMPUTE IDLE']
Durations :  [2.59464 2.6207e-05 0.00397741 ... 1.473e-06 3.9167e-05 0.369753]


In [ ]:
print(region_times)
print(region_times["duration"])

# Convert the grouped DataFrame to a dictionary for easier inspection
# Using to_ndarray() to convert Arkouda arrays to NumPy arrays (causes a data transfer)
region_times_dict = {region: duration for region, duration in zip(region_times.index.to_ndarray(), region_times['duration'].to_ndarray())}
region_times = region_times_dict
sorted_region_times = dict(sorted(region_times_dict.items()))
print(sorted_region_times)


DataFrame(['duration'], 36 rows, 2359.00 B)
[0.403048 0.0106835 7.02317 0.418623 0.901253 0.000377857 0.0530395 3.91803 18.8553 10.3983 0.00390294 0.00493636 17.3218 1.86146 0.871765 0.000308416 2.23176 0.351566 1.65775 0.564911 31.6924 0.313341 0.409 0.16032 0.143802 0.000868952 0.532904 0.0151432 72.4297 0.488565 1.53156 0.385582 403.146 0.000444641 0.317491 14.6971]
{'COMPUTE IDLE': 10.3983259739896, 'buildGijCudaKernel': 1.5315646850003186, 'buildKKRMatrixMultiplyKernelCuda': 72.42974503600027, 'copyTMatrixToTauCuda': 0.010683508999878466, 'copyTauToTau00Cuda': 0.004936361000278389, 'copy_info_kernel': 0.0003084160002257974, 'create_pivot_v2<512>': 0.35156551999891184, 'cutlass::Kernel2<cutlass_80_tensorop_z884gemm_32x32_16x4_nn_align1>': 0.5649113660035572, 'getrf_pivot<getrf_params_<double2, 128, 1, 128, 128, 1> >': 0.05303948200019981, 'getrf_pivot<getrf_params_<double2, 256, 2, 512, 64, 4> >': 0.48856537000192013, 'getrf_pivot<getrf_params_<double2, 256, 2, 512, 64, 8> >': 2.23

In [ ]:
import numpy as np
import xarray as xr
# Vectorized energy calculation
power_data_indices = xp.searchsorted(xp.asarray(ak_df_gpupower['time']), xp.asarray(valid_enter_times), side="right") # Bug in searchsorted. Meaning of side="left" and side="right" is swapped
power_data_indices = xp.where(power_data_indices <= 0, len(ak_df_gpupower['time']) - 1 - power_data_indices, power_data_indices) # Indices less than 0 wrap around automatically, this seems to be technically incorrect, but I'm not a data scientist
power_data_indices = power_data_indices._array
print("Power Data Indices: ", power_data_indices, power_data_indices.shape)
last_tracked_powers = ak_df_gpupower['values'][power_data_indices]
print("Last Tracked Powers: ", last_tracked_powers, last_tracked_powers.shape)
last_tracked_energies = last_tracked_powers * durations
print("Last Tracked Energies: ", last_tracked_energies, last_tracked_energies.shape, last_tracked_energies.dtype)

Power Data Indices:  [5904 20 20 ... 5901 5901 5901] (1202689,)
Last Tracked Powers:  [138.29 120.514 120.514 ... 148.836 148.836 148.836] (1202689,)
Last Tracked Energies:  [358.813 0.00315831 0.479334 ... 0.000219235 0.00582946 55.0326] (1202689,) float64


In [ ]:
# Create a mask for the time intervals
# This takes so much longer than pandas, there must be something that Arkouda isn't optimizing
# And it get's even slower as the number of locales increases
energy_mask = (df_power['time'] >= valid_enter_times[:, None]) & (df_power['time'] <= valid_leave_times[:, None])
print("Energy Mask: ", energy_mask)
print(energy_mask.shape)
print(energy_mask[0])

In [ ]:
# This is SOO inefficient, that it makes me want to cry
# My job on hotlum got auto-killed based on the walltime.
energies = []
for i in range(energy_mask.shape[0]):
    power_data = df_power[(df_power['time'] >= valid_enter_times[i]) & (df_power['time'] <= valid_leave_times[i])]
    energy = np.trapz(power_data['values'].to_ndarray(), power_data['time'].to_ndarray()) if not power_data.empty else last_tracked_energies[i]
    energies.append(energy)


In [ ]:
energy_data = {
    'time': valid_enter_times,
    'event': ['Enter'] * len(valid_enter_times),
    'region': valid_regions,
    'energy': last_tracked_energies # THis should be energies, but it takes too long to compute
}

df_energies = ak.DataFrame(energy_data)
region_energy_groupby = df_energies.GroupBy('region', use_series=True)
region_energy_sum = region_energy_groupby.sum('energy').sort_values('energy', ascending=False)
region_energy_sum = region_energy_sum[region_energy_sum['energy'] > 1.0]
print("Region Energy Sum: \n", region_energy_sum.__repr__())

# end_time = time.time()
# runtime = end_time - start_time
# print(f"Runtime: {runtime:.2f} seconds")


region_energy_sum = region_energy_sum.to_pandas(retain_index=True)
print("Region Energy Sum Pandas: \n", region_energy_sum)

for _, energy in region_energy_sum.items():
    print(type(energy))

# Output the regions and their total energies
print("\n=== Total Energy per Region ===")
for region, energy in region_energy_sum['energy'].items():
    total_time = region_times[region]
    print(f"Region: {region}, Total Energy: {energy:.2f} J, Total Time: {total_time:.2f} s")

top_8_regions = region_energy_sum.head(8).index.tolist()
top_8_regions = [ak_df_gpupower[ak_df_gpupower['region'] == region].copy() for region in top_8_regions] # Needs fixing since ak_df_gpupower is not a pandas DataFrame, it's arkouda

print(region_energy_sum.head(8))
print(top_8_regions)
print(region_times)

Region Energy Sum: 
                                                                          energy
sm90_xmma_gemm_cf64cf64_f64f64_cf64_nn_n_tilesize64x64x32_stag...  61687.544917
buildKKRMatrixMultiplyKernelCuda                                   10698.710494
sm90_xmma_gemm_cf64cf64_f64f64_cf64_nn_n_tilesize32x32x32_stag...   4816.837298
getrf_pivot<getrf_params_<double2, 512, 2, 512, 64, 16> >           2876.516369
getrf_pivot<getrf_params_<double2, 512, 2, 512, 32, 32> >           2657.974645
trsm_left_kernel<int, double2, 256, 4, true, false, false, fal...   2239.845253
COMPUTE IDLE                                                        1514.310634
getrf_pivot<getrf_params_<double2, 512, 2, 512, 64, 8> >            1062.302644
ipiv_lower_diag<double2, 512>                                        597.237470
getrf_pivot<getrf_params_<double2, 256, 2, 512, 64, 8> >             334.475123
trsm_left_kernel<int, double2, 256, 4, true, false, false, tru...    276.002138
laswp_kernel2<doubl

## Missing features in arkouda

1. Reading only select columns from parquet files.
    Instead of being ablet to do
    ```py
    df_events = pd.read_parquet(dir+power_cap200, columns=['time', 'metric', 'values', 'region','location','event'])
    ```
    we have to do:

    ```py
    ak_df = ak.read_parquet(dir+power_cap200)

    keys_to_keep = ['time', 'metric', 'values', 'region', 'location', 'event']
    ak_df = {key: ak_df[key] for key in keys_to_keep}
    ak_df = ak.DataFrame(ak_df)
    ```

2. `np.trapezoid`: Needed to be implemented

3. `ak.Series` does not have `groupby`.
    I wanted to do:

    ```py
    region_times = ak.Series(durations, index=valid_regions).groupby(level=0).sum()
    ```

    Since durations is a 1D array, it's the most logical way to represent this where I want to sum the durations grouped by the regions.

    However, since `ak.Series` does not have `groupby`, I have to use `ak.Dataframe` instead.

    ```py
    # Create an Arkouda DataFrame
    df = ak.DataFrame({
        'region': valid_regions,
        'duration': durations
    })

    # Group by 'region' and sum the 'duration'
    df = df.groupby('region').sum("duration")
    ```
5. `xp.diff` is broken when called without an `axis` argument because the default of `-1` to specify the innermost axis does not work in Chapel.
6. `xp.sum` is broken when called with negative `axis` argument because that does not work in Chapel.
7. For some reason `xp.trapz` always returns an `<class 'arkouda.array_api.array_object.Array'>` even when it' supposed to return a single value.

This caused issues with something as simple as:
```
        val = xp.trapz(xp.asarray(power_data['values']), xp.asarray(power_data['time'])) #if not power_data.empty else last_tracked_energies[i]
        print(val)
        print(type(val))
        print(type(val._array))
        energies[i] = val # error since it tries to convert just a float to_ndarray.

```
8. For a 2D array, I can't do `energies[i]`, I have to do `energies[i, :]`


In [ ]:
for region in df['region'].value_counts()[0]:
    region_slice_of_df = df[df['region'] == region]
    # Calculate enter and leave events
    # calculate durations
    # calculate energy


In [ ]:
ak.shutdown()